In [1]:
import json
from pathlib import Path
from IPython.display import display, HTML
import ipywidgets as W
import ipyfilechooser as FC

In [2]:
CONFIG_PATH = Path("sim_config_step1.json")

In [3]:
mode_toggle = W.ToggleButtons(
    options=[("Standard user", "standard"), ("Expert user", "expert")],
    value="standard",
    description="Mode:"
)

In [4]:
pkg_location = FC.FileChooser(
    title="Select package location",
    description="Package location:",
    show_hidden=False,
    select_default=True,
    use_dir_icons=True,
    width="100%"
)
pkg_location.default_filename = "my_simulation_package"
pkg_location.register_callback(
    lambda chooser: print(f"Selected package location: {chooser.selected_path}")
)

display(mode_toggle, pkg_location)

ToggleButtons(description='Mode:', options=(('Standard user', 'standard'), ('Expert user', 'expert')), value='…

FileChooser(path='/home/flo/repos/SystemSimulation/examples/UI', filename='my_simulation_package', title='Sele…

In [6]:
pkg_path = Path(pkg_location.value)

# Get files in package
pkg_dir = pkg_path.parent

# Get all .mo files in the package directory
mo_files = list(pkg_dir.glob("*.mo"))
print(f"Found {len(mo_files)} .mo files in {pkg_dir}")
mo_files.sort()

# Show all mo files in a dropdown
mo_file_dropdown = W.Dropdown(
    options=[(f.name, f) for f in mo_files],
    description="Model file:",
    style={"description_width": "initial"},
    layout=W.Layout(width="50%")
)

display(mo_file_dropdown)

# Show the content of the selected .mo file
def show_mo_file_content(change):
    if change["type"] == "change" and change["name"] == "value":
        selected_file = change["new"]
        if selected_file and selected_file.exists():
            with open(selected_file, "r") as f:
                content = f.read()
            mo_file_content.value = content
        else:
            mo_file_content.value = "File does not exist."
mo_file_dropdown.observe(show_mo_file_content)
mo_file_content = W.Textarea(
    value="",
    description="File content:",
    layout=W.Layout(width="100%", height="400px"),
    disabled=True
)
display(mo_file_content)

Found 14 .mo files in /home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum


Dropdown(description='Model file:', layout=Layout(width='50%'), options=(('AngleEncoder.mo', PosixPath('/home/…

Textarea(value='', description='File content:', disabled=True, layout=Layout(height='400px', width='100%'))

In [27]:
# Widget to display selected models
relevant_models = W.SelectMultiple(
    options=[f.name for f in mo_files],
    description="Relevant models:",
    style={"description_width": "initial"},
    layout=W.Layout(width="50%", height="150px")
)

selected_models = W.Select(
    options=[],
    description="Selected models:",
    style={"description_width": "initial"},
    layout=W.Layout(width="50%", height="150px")
)

def update_selected_models(change):
    selected_models.options = list(change["new"])

relevant_models.observe(update_selected_models, names="value")

display(relevant_models, selected_models)

SelectMultiple(description='Relevant models:', layout=Layout(height='150px', width='50%'), options=('Demo_Driv…

Select(description='Selected models:', layout=Layout(height='150px', width='50%'), options=(), style=Descripti…

In [ ]:


pkg_location = W.Text(
    value="",   
    description="Package Path:",
    placeholder="Optional: path to Modelica package directory",
    layout=W.Layout(width="60%"),
)
model_file = W.Text(
    value="",
    description="Model File:",
    placeholder="Path to .mo file (e.g., /path/to/MySystem.mo)",
    layout=W.Layout(width="60%"),
)
model_class = W.Text(
    value="",
    description="Model Class:",
    placeholder="e.g., MyLib.Subsystems.Pendulum",
    layout=W.Layout(width="60%"),
)

use_mode = W.Dropdown(
    options=[("Run as Modelica (direct)", "modelica"), ("Export as FMU", "fmu")],
    value="modelica",
    description="Use Model:",
)

fmi_version = W.Dropdown(
    options=[("FMI 3.0", "3.0"), ("FMI 2.0", "2.0")],
    value="3.0",
    description="FMI Version:",
)
fmu_type = W.Dropdown(
    options=[("Model Exchange (ME)", "ME"), ("Co-Simulation (CS)", "CS")],
    value="ME",
    description="FMU Type:",
)
binary_format = W.Dropdown(
    options=[("Native", "native"), ("x86_64", "x86_64"), ("arm64", "arm64")],
    value="native",
    description="Binary:",
)
compiler_flags = W.Textarea(
    value="",
    placeholder="Extra compiler flags (optional)",
    description="Compiler Flags:",
    layout=W.Layout(width="60%", height="60px"),
)

expert_box = W.VBox([
    W.HTML("<b>Advanced FMU Options</b>"),
    W.HBox([fmi_version, fmu_type, binary_format]),
    compiler_flags,
])

status = W.HTML("<i>Fill in the fields to continue…</i>", layout=W.Layout(margin="8px 0"))
problems = W.HTML("", layout=W.Layout(margin="0 0 8px 0"))

def validate():
    msgs = []
    if not model_file.value.strip():
        msgs.append("• Missing <b>Model File</b> (path to .mo).")
    if not model_class.value.strip():
        msgs.append("• Missing <b>Model Class</b> (e.g., MyLib.Subsystems.Pendulum).")
    if msgs:
        problems.value = "<div style='color:#b91c1c'>" + "<br>".join(msgs) + "</div>"
        status.value = "<span style='color:#b45309'>Please complete the required fields.</span>"
        return False
    else:
        problems.value = ""
        status.value = "<span style='color:#065f46'>Looks good. You can save this step.</span>"
        return True

for w in [pkg_location, model_file, model_class, use_mode, fmi_version, fmu_type, binary_format, compiler_flags]:
    w.observe(lambda change: validate(), names="value")

def on_mode_change(change):
    expert_box.layout.display = "none" if mode_toggle.value == "standard" else "block"
    validate()
mode_toggle.observe(on_mode_change, names="value")
on_mode_change(None)

save_btn = W.Button(description="Save Step 1", icon="save", button_style="success")
save_out = W.Output()

def on_save_clicked(b):
    save_out.clear_output()
    if not validate():
        with save_out:
            display(HTML("<div style='color:#b91c1c'>Cannot save: please fix the issues above.</div>"))
        return
    cfg = {
        "user_mode": mode_toggle.value,
        "model": {
            "package_path": pkg_location.value.strip() or None,
            "model_file": model_file.value.strip(),
            "model_class": model_class.value.strip(),
            "use_mode": use_mode.value,
        },
        "fmu_options": None,
    }
    if use_mode.value == "fmu":
        cfg["fmu_options"] = {
            "fmi_version": fmi_version.value,
            "fmu_type": fmu_type.value,
            "binary": binary_format.value,
            "compiler_flags": compiler_flags.value.strip() or None,
        }
    CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
    with save_out:
        display(HTML(
            f"<div style='color:#065f46'>Saved Step 1 configuration.</div>"
            f"<div>Saved: <code>{CONFIG_PATH.resolve()}</code></div>"
        ))
save_btn.on_click(on_save_clicked)

panel = W.VBox([
    W.HTML("<h3>Step 1 — Choose Modelica Model & Usage</h3>"),
    mode_toggle,
    W.HTML("<hr>"),
    W.HTML("<b>Modelica Source</b>"),
    pkg_location,
    model_file,
    model_class,
    W.HTML("<b>Usage</b>"),
    use_mode,
    expert_box,
    W.HTML("<hr>"),
    status,
    problems,
    W.HBox([save_btn]),
    save_out,
])

display(panel)

## 2) Next steps (suggested)

- **Step 2 — Solver & Simulation Settings**: ask for global time settings, tolerance, stop time; in Expert mode expose solver choices, step-size policy, event handling.
- **Step 3 — Build/Export**: if FMU selected, run OM → FMU export and show logs. If Modelica-only, compile and prepare simulation.
- **Step 4 — Run**: execute simulation with a progress log; offer live plots for selected signals.
- **Step 5 — Results**: store a run folder with metadata (config JSON/YAML, logs), plots, and CSV/HDF5 outputs.